# Factor Model: Step 4b - Alpha 179 Calculation

## Objective
Calculate Alpha 179 factors and save results separately.

### Workflow:
1. Load data panels from Step 4a
2. Load Alpha 179 (Alpha191) class
3. Calculate all 179 alpha factors
4. **Save Alpha 179 results to disk**

### Inputs:
- `data_panels.parquet` - From Step 4a

### Outputs:
- `alpha179_results.parquet` - Successfully calculated factors
- `alpha179_errors.csv` - Errors log

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('.')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Load Data Panels from Step 4a

In [ ]:
print("Loading data panels from Step 4a...")

panels_combined = pd.read_parquet('data_panels.parquet')

print(f"✓ Data panels loaded")
print(f"  Shape: {panels_combined.shape}")
print(f"  Columns: {panels_combined.columns.tolist()}")
print(f"  Memory usage: {panels_combined.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Unstack to panel format (date × symbol)
print("\nConverting to panel format (date × symbol)...")
print("  ⚠ This may take a few minutes for large datasets...")

# Optimize memory by converting to float32
print("  Optimizing data types to reduce memory...")
for col in ['open', 'high', 'low', 'close', 'volume', 'returns']:
    if col in panels_combined.columns:
        panels_combined[col] = panels_combined[col].astype('float32')

open_panel = panels_combined['open'].unstack()
high_panel = panels_combined['high'].unstack()
low_panel = panels_combined['low'].unstack()
close_panel = panels_combined['close'].unstack()
volume_panel = panels_combined['volume'].unstack()
returns_panel = panels_combined['returns'].unstack()

# Handle vwap/avg_price (Step 4a saves it as 'vwap')
if 'vwap' in panels_combined.columns:
    panels_combined['vwap'] = panels_combined['vwap'].astype('float32')
    vwap_panel = panels_combined['vwap'].unstack()
elif 'avg_price' in panels_combined.columns:
    panels_combined['avg_price'] = panels_combined['avg_price'].astype('float32')
    vwap_panel = panels_combined['avg_price'].unstack()
else:
    print("  ⚠ Warning: No vwap or avg_price found, calculating from OHLC")
    vwap_panel = ((high_panel + low_panel + close_panel) / 3).astype('float32')

# Clean up to free memory
del panels_combined
import gc
gc.collect()

print(f"✓ Panels unstacked")
print(f"  Panel shape: {close_panel.shape}")
print(f"  Dates: {len(close_panel)}")
print(f"  Symbols: {len(close_panel.columns)}")
print(f"  Memory per panel: ~{close_panel.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## 2. Create Data Dictionary

In [ ]:
# Create data dictionary for Alpha179
data = {
    'open': open_panel,
    'high': high_panel,
    'low': low_panel,
    'close': close_panel,
    'volume': volume_panel,
    'vwap': vwap_panel,
    'returns': returns_panel
}

print("✓ Data dictionary created for Alpha179")
print(f"  Keys: {list(data.keys())}")
print(f"  Sample shape (close): {data['close'].shape}")

## 3. Load Alpha 179 (Alpha191) Class

**Note**: Using dedicated loader to avoid Jupyter kernel caching issues.

In [ ]:
print("="*80)
print("LOADING ALPHA 179 CLASS")
print("="*80)

# Use dedicated loader to avoid kernel caching issues
from alpha179_loader import get_alpha179_instance

try:
    # Load and instantiate Alpha191
    alpha179 = get_alpha179_instance(data)
    
    print("\n✓ Alpha179 ready for calculations")
    
except Exception as e:
    print(f"\n❌ ERROR loading Alpha179: {e}")
    print("\nTroubleshooting:")
    print("  1. Make sure alpha179_factors.ipynb exists in this directory")
    print("  2. Make sure alpha179_loader.py exists in this directory")
    print("  3. Try restarting the kernel")
    
    alpha179 = None

## 4. Calculate Alpha 179 Factors

In [ ]:
print("="*80)
print("CALCULATING ALPHA 179 FACTORS")
print("="*80)
print("\nThis will take several minutes...\n")

alpha179_results = {}
alpha179_errors = {}

if alpha179 is None:
    print("⚠ Alpha179 object not initialized - skipping calculations")
    for i in range(1, 192):
        alpha179_errors[f'alpha179_{i:03d}'] = "Alpha179 class not initialized"
else:
    for i in tqdm(range(1, 192), desc="Alpha 179"):
        method_name = f'alpha_{i:03d}'
        
        try:
            if hasattr(alpha179, method_name):
                alpha_method = getattr(alpha179, method_name)
                result = alpha_method()
                
                if isinstance(result, (pd.Series, pd.DataFrame)) and len(result) > 0:
                    alpha179_results[f'alpha179_{i:03d}'] = result
                elif result == 0:
                    alpha179_errors[f'alpha179_{i:03d}'] = "Not implemented (returns 0)"
                else:
                    alpha179_errors[f'alpha179_{i:03d}'] = "Empty result"
            else:
                alpha179_errors[f'alpha179_{i:03d}'] = "Method not found"
                
        except Exception as e:
            alpha179_errors[f'alpha179_{i:03d}'] = str(e)

print(f"\n✓ Alpha 179 calculation complete")
print(f"  Successful: {len(alpha179_results)}/191")
print(f"  Errors/Missing: {len(alpha179_errors)}")

if alpha179_errors:
    print(f"\n⚠ Errors/Missing factors (first 10):")
    for name, error in list(alpha179_errors.items())[:10]:
        print(f"  {name}: {error[:60]}...")

## 5. Save Alpha 179 Results

In [ ]:
print("="*80)
print("SAVING ALPHA 179 RESULTS")
print("="*80)

if len(alpha179_results) > 0:
    print(f"\nSaving {len(alpha179_results)} Alpha 179 factors...")
    
    # Convert to DataFrame format
    alpha179_dfs = []
    
    for factor_name, factor_values in alpha179_results.items():
        if isinstance(factor_values, pd.Series):
            df_temp = factor_values.to_frame(name=factor_name)
        elif isinstance(factor_values, pd.DataFrame):
            if len(factor_values.columns) > 1:
                df_temp = factor_values.stack().to_frame(name=factor_name)
            else:
                df_temp = factor_values.rename(columns={factor_values.columns[0]: factor_name})
        alpha179_dfs.append(df_temp)
    
    df_alpha179 = pd.concat(alpha179_dfs, axis=1)
    
    # Save to parquet
    df_alpha179.to_parquet('alpha179_results.parquet', compression='snappy')
    
    import os
    file_size_mb = os.path.getsize('alpha179_results.parquet') / 1024**2
    print(f"✓ Alpha 179 results saved to alpha179_results.parquet")
    print(f"  Size: {file_size_mb:.2f} MB")
    print(f"  Shape: {df_alpha179.shape}")
else:
    print("❌ No Alpha 179 results to save!")

# Save errors log
if alpha179_errors:
    error_df = pd.DataFrame([
        {'factor': k, 'error': v} for k, v in alpha179_errors.items()
    ])
    error_df.to_csv('alpha179_errors.csv', index=False)
    print(f"\n✓ Error log saved to alpha179_errors.csv")
    print(f"  {len(alpha179_errors)} errors logged")

## Summary

In [ ]:
print("="*80)
print("STEP 4B COMPLETE - ALPHA 179 CALCULATED")
print("="*80)

print(f"\nResults:")
print(f"  Successfully calculated: {len(alpha179_results)}/191")
print(f"  Errors/Missing: {len(alpha179_errors)}")
print(f"  Success rate: {len(alpha179_results)/191*100:.1f}%")

print(f"\nOutput files:")
if len(alpha179_results) > 0:
    print(f"  ✓ alpha179_results.parquet - Alpha 179 factor loadings")
if alpha179_errors:
    print(f"  ✓ alpha179_errors.csv - Error log")

print(f"\n" + "="*80)
print("Next: Run Step 4c to combine all factors and standardize")
print("="*80)